0. Lembrar de ativar GPUs: T4 no ambiente de execução

In [1]:
# 0. Instalação e sincronização das dependências do ecossistema Hugging Face, LIME

!pip install -U "datasets>=2.14.0" "transformers>=4.30.0" "accelerate>=0.20.0" "pyarrow>=14.0.0" kagglehub lime scikit-learn -q

1. Carregamento do Dataset via Kaggle Hub

In [2]:
import kagglehub
import os
import pandas as pd

# Download do dataset pré-processado do meu repositório público

path = kagglehub.dataset_download("angelosbc/steam-reviews-in-portuguese-pt-br-2021")
print("Pasta do dataset:", path)

# Carrega o CSV filtrado no Pandas
caminho_csv = os.path.join(path, "steam_reviews_brazilian.csv")
df = pd.read_csv(caminho_csv)

# Mostra total de avaliações e alguns dados
print(f"Total de avaliações em português: {len(df):,}")
df.head()

Using Colab cache for faster access to the 'steam-reviews-in-portuguese-pt-br-2021' dataset.
Pasta do dataset: /kaggle/input/steam-reviews-in-portuguese-pt-br-2021
Total de avaliações em português: 918,910


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,29,292030,The Witcher 3: Wild Hunt,85177505,brazilian,Se um dia alguém falar que esse jogo é ruim na...,1611368498,1611368498,True,1,...,True,False,False,76561198844659805,70,4,11115.0,2252.0,11115.0,1.611186e+09
1,30,292030,The Witcher 3: Wild Hunt,85176839,portuguese,bom demais\n,1611367482,1611367482,True,0,...,True,False,False,76561198847379347,4,1,555.0,465.0,555.0,1.611367e+09
2,32,292030,The Witcher 3: Wild Hunt,85176661,brazilian,NaN,1611367193,1611367193,True,0,...,True,False,False,76561198076880796,127,13,875.0,752.0,826.0,1.611370e+09
3,34,292030,The Witcher 3: Wild Hunt,85176249,brazilian,Obra prima!!!,1611366524,1611366524,True,0,...,True,False,False,76561198957873353,32,1,2888.0,1475.0,2888.0,1.611366e+09
4,43,292030,The Witcher 3: Wild Hunt,85173023,brazilian,Jogão da porra.,1611361229,1611361229,True,0,...,True,False,False,76561198141110905,59,4,20193.0,3692.0,20193.0,1.611297e+09


2. Pré-processamento Textual e Divisão dos Dados

In [3]:
import re
from sklearn.model_selection import train_test_split

# 1. Remove linhas nulas nas colunas essenciais
print("-> Removendo valores ausentes...")
df = df.dropna(subset=['review', 'recommended'])

# 2. Converte a coluna alvo para inteiro (0 = Negativo, 1 = Positivo)
df['label'] = df['recommended'].astype(int)

# 3. Função de normalização e limpeza sintática do texto
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Converte para minúsculas
    texto = texto.lower()
    # Remove URLs completas
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # Remove tags HTML
    texto = re.sub(r'<.*?>', '', texto)
    # Mantém apenas letras acentuadas e espaços, removendo pontuações/símbolos
    texto = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', ' ', texto)
    # Remove espaços em branco redundantes
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 4. Aplica a limpeza e filtra textos com menos de 4 caracteres
print("-> Aplicando rotina de limpeza no texto...")
df['clean_review'] = df['review'].apply(limpar_texto)
df = df[df['clean_review'].str.len() >= 4]

print(f"Total de registros válidos pós-limpeza: {len(df):,}")

# 5. Divisão estratificada (70% Treino, 15% Validação, 15% Teste)
print("-> Realizando divisão estratificada (70/15/15)...")
X = df['clean_review']
y = df['label']

# Primeiro corte: separa 70% treino e 30% temporário
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Segundo corte: divide os 30% temporários igualmente entre validação e teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Conjunto de Treino:     {len(X_train):,} amostras")
print(f"Conjunto de Validação:  {len(X_val):,} amostras")
print(f"Conjunto de Teste:      {len(X_test):,} amostras")

-> Removendo valores ausentes...
-> Aplicando rotina de limpeza no texto...
Total de registros válidos pós-limpeza: 853,982
-> Realizando divisão estratificada (70/15/15)...
Conjunto de Treino:     597,787 amostras
Conjunto de Validação:  128,097 amostras
Conjunto de Teste:      128,098 amostras


3. Balanceamento do Conjunto de Treino

In [4]:
# 1. Reconstrói o DataFrame com o conjunto de treino bruto
df_train = pd.DataFrame({
    'clean_review': X_train,
    'label': y_train
})

# 2. Separa as classes positiva e negativa
df_pos = df_train[df_train['label'] == 1]
df_neg = df_train[df_train['label'] == 0]

# 3. Cria a amostra balanceada (35.000 positivas + 35.000 negativas = 70k)
n_samples = min(len(df_neg), 35000)

df_train_sample = pd.concat([
    df_pos.sample(n=n_samples, random_state=42),
    df_neg.sample(n=n_samples, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"-> Amostra balanceada de treino criada com sucesso!")
print(f"Total de instâncias para treino: {len(df_train_sample):,}")
print("\nDistribuição das classes no treino:")
print(df_train_sample['label'].value_counts())

-> Amostra balanceada de treino criada com sucesso!
Total de instâncias para treino: 70,000

Distribuição das classes no treino:
label
0    35000
1    35000
Name: count, dtype: int64


4. Baseline (TF-IDF + Regressão Logística)




In [5]:
##########
##TF-IDF##
##########


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

# 1. Vetorização TF-IDF treinada exclusivamente na amostra balanceada
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(df_train_sample['clean_review'])
X_test_vec = tfidf.transform(X_test)

# 2. Treinamento da Regressão Logística
clf_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_lr.fit(X_train_vec, df_train_sample['label'])

# 3. Predições sobre o conjunto de teste de 128k
y_pred_lr = clf_lr.predict(X_test_vec)
y_prob_lr = clf_lr.predict_proba(X_test_vec)[:, 1]

print("="*50)
print(" RESULTADOS: TF-IDF + REGRESSÃO LOGÍSTICA (TREINO BALANCEADO)")
print("="*50)
print(f"Acurácia:        {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score Macro: {f1_score(y_test, y_pred_lr, average='macro'):.4f}")
print(f"ROC-AUC:         {roc_auc_score(y_test, y_prob_lr):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred_lr, target_names=['Não Recomenda', 'Recomenda']))

 RESULTADOS: TF-IDF + REGRESSÃO LOGÍSTICA (TREINO BALANCEADO)
Acurácia:        0.9034
F1-Score Macro: 0.7344
ROC-AUC:         0.9604

Relatório de Classificação Detalhado:
               precision    recall  f1-score   support

Não Recomenda       0.37      0.90      0.52      7503
    Recomenda       0.99      0.90      0.95    120595

     accuracy                           0.90    128098
    macro avg       0.68      0.90      0.73    128098
 weighted avg       0.96      0.90      0.92    128098



5. Download do GloVe, Tokenização e Matriz de Embeddings

In [6]:
import os
import zipfile
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Download dos embeddings GloVe (NILC)
glove_txt = "glove_s100.txt"

if not os.path.exists(glove_txt):
    print("-> Baixando GloVe 100d...")
    !wget -q --show-progress "https://huggingface.co/datasets/liaad/glove-pt-br/resolve/main/glove_s100.txt" -O glove_s100.txt || true
    if not os.path.exists(glove_txt):
        !wget -q --show-progress "https://object.c3s.uni-duesseldorf.de/nilc/glove_s100.zip" -O glove_s100.zip && unzip -q glove_s100.zip || true

# 2. Tokenizacao e sequenciamento (T = 120)
MAX_LEN = 120
MAX_WORDS = 50_000
EMBEDDING_DIM = 100

print("-> Ajustando tokenizer nos dados de treino...")
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding='post', truncating='post')

# 3. Construcao da matriz de pesos
print("-> Montando matriz de embeddings...")
embeddings_index = {}
with open(glove_txt, encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

num_words_matrix = min(MAX_WORDS, len(tokenizer.word_index) + 1)
embedding_matrix = np.zeros((num_words_matrix, EMBEDDING_DIM))

for word, i in tokenizer.word_index.items():
    if i < MAX_WORDS:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

print(f"-> Matriz de embeddings construida: {embedding_matrix.shape}")

-> Ajustando tokenizer nos dados de treino...
-> Montando matriz de embeddings...
-> Matriz de embeddings construida: (50000, 100)


6. BiLSTM

In [7]:
####$$$$
#BiLSTM#
########

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

# 1. Parâmetros de Sequência e Vocabulário
MAX_VOCAB_SIZE = 20000
MAX_LEN = 128
EMBEDDING_DIM = 100

# 2. Tokenização e Padding
tokenizer_lstm = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer_lstm.fit_on_texts(df_train_sample['clean_review'])

X_train_seq = pad_sequences(tokenizer_lstm.texts_to_sequences(df_train_sample['clean_review']), maxlen=MAX_LEN)
# Amostra de validação mais leve (10k) para não travar entre as épocas
X_val_sample_text = list(X_val)[:10000]
y_val_sample_arr  = np.array(list(y_val)[:10000])
X_val_seq   = pad_sequences(tokenizer_lstm.texts_to_sequences(X_val_sample_text), maxlen=MAX_LEN)

# Teste completo de 128k
X_test_seq  = pad_sequences(tokenizer_lstm.texts_to_sequences(X_test), maxlen=MAX_LEN)

y_train_arr = np.array(df_train_sample['label'])
y_test_arr  = np.array(list(y_test))

# 3. Arquitetura BiLSTM com CuDNN ativado (sem recurrent_dropout)
model_bilstm = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, return_sequences=False, dropout=0.2)), # Ativa aceleração total na GPU
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_bilstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                     loss='binary_crossentropy',
                     metrics=['accuracy'])

# 4. Treinamento com Early Stopping
callbacks = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]

print("-> Treinando BiLSTM com aceleração CuDNN...")
history = model_bilstm.fit(
    X_train_seq, y_train_arr,
    validation_data=(X_val_seq, y_val_sample_arr),
    epochs=5,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

# 5. Avaliação no Teste Real de 128k
print("\n-> Avaliando no conjunto de teste de 128k...")
y_probs_lstm = model_bilstm.predict(X_test_seq, batch_size=512).flatten()
y_preds_lstm = (y_probs_lstm >= 0.5).astype(int)

print("="*50)
print(" RESULTADOS: BiLSTM (TREINO BALANCEADO)")
print("="*50)
print(f"Acurácia:        {accuracy_score(y_test_arr, y_preds_lstm):.4f}")
print(f"F1-Score Macro: {f1_score(y_test_arr, y_preds_lstm, average='macro'):.4f}")
print(f"ROC-AUC:         {roc_auc_score(y_test_arr, y_probs_lstm):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test_arr, y_preds_lstm, target_names=['Não Recomenda', 'Recomenda']))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


-> Treinando BiLSTM com aceleração CuDNN...
Epoch 1/5
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.8509 - loss: 0.3469 - val_accuracy: 0.8971 - val_loss: 0.2516
Epoch 2/5
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9132 - loss: 0.2348 - val_accuracy: 0.8997 - val_loss: 0.2643
Epoch 3/5
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9263 - loss: 0.2015 - val_accuracy: 0.8800 - val_loss: 0.3108

-> Avaliando no conjunto de teste de 128k...
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step
 RESULTADOS: BiLSTM (TREINO BALANCEADO)
Acurácia:        0.8961
F1-Score Macro: 0.7226
ROC-AUC:         0.9587

Relatório de Classificação Detalhado:
               precision    recall  f1-score   support

Não Recomenda       0.35      0.90      0.50      7503
    Recomenda       0.99      0.90      0.94    120595

     accuracy                           0.90    128098
    macro avg       0.67      0.90      0.72    128098
 weighted avg       0.96      0.90      0.92    128098

7. Fine-Tuning do Modelo BERTimbau

In [8]:
###########
#BERTimbau#
###########

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 1. Configuração do dispositivo acelerador
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilizando dispositivo: {device}")

# 2. Tokenização com o modelo BERTimbau
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_func(examples):
    # max_length=96 otimiza o tempo de atenção sem perda nas reviews da Steam
    return tokenizer(examples['clean_review'], padding="max_length", truncation=True, max_length=96)

# Converte DataFrames existentes em Datasets do Hugging Face
print("-> Preparando Datasets para o BERTimbau...")
train_dataset = Dataset.from_pandas(df_train_sample[['clean_review', 'label']]).map(tokenize_func, batched=True)

val_df = pd.DataFrame({'clean_review': list(X_val), 'label': list(y_val)})
val_dataset = Dataset.from_pandas(val_df).map(tokenize_func, batched=True)

test_df = pd.DataFrame({'clean_review': list(X_test), 'label': list(y_test)})
test_dataset = Dataset.from_pandas(test_df).map(tokenize_func, batched=True)

# 3. Carregamento da arquitetura Transformer
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# 4. Função para computar métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(labels, probs)
    return {"accuracy": acc, "f1_macro": f1, "roc_auc": auc}

# 5. Hiperparâmetros otimizados (fp16 habilitado e avaliação por época)
training_args = TrainingArguments(
    output_dir="./bertimbau_steam_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none"
)

# 6. Instanciação do pipeline de treino
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# 7. Execução do fine-tuning
print("\n-> Iniciando Fine-Tuning do BERTimbau...")
trainer.train()

# 8. Avaliação final sobre o conjunto de teste completo (128k)
print("\n-> Avaliando no conjunto de teste...")
predictions = trainer.predict(test_dataset)
logits = predictions.predictions
y_probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
y_preds = np.argmax(logits, axis=1)
y_true = np.array(list(y_test))

print("\n" + "="*50)
print(" RESULTADOS: TRANSFORMER (BERTimbau)")
print("="*50)
print(f"Acurácia:        {accuracy_score(y_true, y_preds):.4f}")
print(f"F1-Score Macro:  {f1_score(y_true, y_preds, average='macro'):.4f}")
print(f"ROC-AUC:         {roc_auc_score(y_true, y_probs):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_true, y_preds, target_names=['Não Recomenda', 'Recomenda']))

Utilizando dispositivo: cuda


-> Preparando Datasets para o BERTimbau...


Map:   0%|          | 0/70000 [00:00<?, ? examples/s]

Map:   0%|          | 0/128097 [00:00<?, ? examples/s]

Map:   0%|          | 0/128098 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Roc Auc
1,0.226380,0.181020,0.933621,0.789764,0.973514
2,0.163737,0.205841,0.927477,0.778973,0.974175


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-> Avaliando no conjunto de teste...



 RESULTADOS: TRANSFORMER (BERTimbau)
Acurácia:        0.9346
F1-Score Macro:  0.7910
ROC-AUC:         0.9736

Relatório de Classificação Detalhado:
               precision    recall  f1-score   support

Não Recomenda       0.47      0.90      0.62      7503
    Recomenda       0.99      0.94      0.96    120595

     accuracy                           0.93    128098
    macro avg       0.73      0.92      0.79    128098
 weighted avg       0.96      0.93      0.94    128098

